# Lab 6: Makemore - Bigram Language Model

## 🎯 Learning Objectives

By the end of this lab, you will:
- Understand **tokenization** deeply - why we need it and how it works
- See how **loss functions act as the teacher** that guides model learning
- Build a **bigram model** from scratch (predicting next character from current)
- **Visualize the learned bigram table** and interpret patterns
- **Generate text** from a simple statistical model

**What You'll Build:** A character-level bigram model that learns which letters commonly follow each other in names

**Dataset:** Names dataset (~32,000 names from [Karpathy's makemore](https://github.com/karpathy/makemore/blob/master/names.txt))

**Why bigrams?** They're the simplest language model! Perfect for understanding the fundamentals before moving to more complex architectures.

**What is a bigram?**
- "bi" = two, "gram" = sequence
- Predicting next character based on current character
- Example: After 'q', we almost always see 'u'!
- After 'a', we might see 'n', 'l', 'r', or 'x'

**Note:** PyTorch is pre-installed in Google Colab!

## Part 0: Setup and Dataset

Let's download the names dataset and explore it

In [7]:
import sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print("✓ All imports successful!")

✓ All imports successful!


In [8]:
# Download dataset if in Colab
import os

if 'google.colab' in sys.modules:
    if not os.path.exists('names.txt'):
        print("Downloading names.txt...")
        !wget -q https://raw.githubusercontent.com/karpathy/makemore/master/names.txt
        print("✓ Download complete!")
    else:
        print("✓ names.txt already exists")
else:
    print("Running locally - ensure names.txt is in current directory")

# Load names
with open('names.txt', 'r', encoding='utf-8') as f:
    names = f.read().splitlines()

# Show statistics
max_name_length = max(len(name) for name in names)
avg_name_length = sum(len(name) for name in names) / len(names)

print(f"\nDataset statistics:")
print(f"  Total names: {len(names):,}")
print(f"  Max name length: {max_name_length}")
print(f"  Avg name length: {avg_name_length:.1f}")
print(f"\nFirst 20 names:")
print(', '.join(names[:20]))

Running locally - ensure names.txt is in current directory

Dataset statistics:
  Total names: 32,033
  Max name length: 15
  Avg name length: 6.1

First 20 names:
emma, olivia, ava, isabella, sophia, charlotte, mia, amelia, harper, evelyn, abigail, emily, elizabeth, mila, ella, avery, sofia, camila, aria, scarlett


## Part 1: Character-Level Tokenizer

### Why Do We Need Tokenization?

Neural networks work with **numbers**, not text. We need to:
1. **Encode**: Convert text → numbers (for the model)
2. **Decode**: Convert numbers → text (to read output)

### Example

- Character level tokenizer
- 
```
Tokens: f, r, i, e, n, d, l, y
Total Tokens: 8
```

- Sub-word level tokenizer

```
Example (using Byte-Pair Encoding): friend, ly
Total Tokens: 2
```

- Word level tokenizer

```
Tokens: friendly
Total Tokens: 1
```

### Special Tokens

We'll add two special tokens:
- `<S>` = Start of name (helps model learn how names begin)
- `<E>` = End of name (helps model learn when to stop generating)

Example:
```
Original: "sam"
With tokens: <S> s a m <E>
```

## Exercise 1: Build Character Tokenizer

### Your Task

Implement a character-level tokenizer with:
1. Build vocabulary from dataset (all unique characters + special tokens)
2. Create `char2id` mapping (character → integer ID)
3. Create `id2char` mapping (integer ID → character)
4. Implement `encode(text)` to convert string → list of IDs
5. Implement `decode(ids)` to convert list of IDs → string

### Hints
- Use `set()` to get unique characters from all names
- Add special tokens FIRST so they get IDs 0 and 1
- Sort the rest alphabetically for consistent ordering
- Use dictionary comprehension: `{char: i for i, char in enumerate(...)}`

In [9]:
class CharTokenizer:
    """Simple character-level tokenizer with special tokens."""

    def __init__(self, names):
        # Get unique characters from all names
        chars_set = set(''.join(names))

        # Add special tokens FIRST
        chars = ['<S>', '<E>'] + sorted(list(chars_set))

        # Store vocabulary size
        self.vocab_size = len(chars)

        # Create char2id mapping
        self.char2id = {ch: i for i, ch in enumerate(chars)}

        # Create id2char mapping
        self.id2char = {i: ch for i, ch in enumerate(chars)}

        # Store special token IDs for easy access
        self.start_id = self.char2id['<S>']
        self.end_id = self.char2id['<E>']

    def encode(self, text):
        """Convert text to list of token IDs."""
        return [self.char2id[ch] for ch in text]

    def decode(self, ids):
        """Convert list of token IDs to text."""
        return ''.join([self.id2char[i] for i in ids])

# Create tokenizer
tokenizer = CharTokenizer(names)
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Special tokens: <S>={tokenizer.start_id}, <E>={tokenizer.end_id}")
print(f"\nFirst 10 characters in vocab:")
for i in range(10):
    print(f"  {i}: '{tokenizer.id2char[i]}'")

Vocabulary size: 28
Special tokens: <S>=0, <E>=1

First 10 characters in vocab:
  0: '<S>'
  1: '<E>'
  2: 'a'
  3: 'b'
  4: 'c'
  5: 'd'
  6: 'e'
  7: 'f'
  8: 'g'
  9: 'h'


### Test Your Tokenizer

In [10]:
# Test encode/decode
test_names = ["sam", "emma", "alex"]

for name in test_names:
    encoded = tokenizer.encode(name)
    decoded = tokenizer.decode(encoded)
    print(f"Original: '{name}'")
    print(f"Encoded:  {encoded}")
    print(f"Decoded:  '{decoded}'")
    assert decoded == name, f"Mismatch: {decoded} != {name}"
    print("✓ Pass\n")

print("✓ Tokenizer works perfectly!")

Original: 'sam'
Encoded:  [20, 2, 14]
Decoded:  'sam'
✓ Pass

Original: 'emma'
Encoded:  [6, 14, 14, 2]
Decoded:  'emma'
✓ Pass

Original: 'alex'
Encoded:  [2, 13, 6, 25]
Decoded:  'alex'
✓ Pass

✓ Tokenizer works perfectly!


## Part 2: What is a Bigram Model?

### Concept

A **bigram** predicts the next character based on the current character.

**Example patterns in English names:**
- After 'q' → almost always 'u'
- After 'a' → often 'n', 'l', 'r', or 'x'
- After 'x' → likely 'a', 'e', or <E> (end of name)

### How It Works

Build a **count table**:
```
         Next char
        a  b  c  d  ...
Curr a [2, 0, 1, 0, ...]
     b [1, 0, 0, 0, ...]
     c [1, 0, 0, 0, ...]
     ...
```

Then convert counts → probabilities:
```
Count table:    [2, 0, 1, 0]  (after 'a', we saw: 2×b, 0×c, 1×d, 0×e)
Total:          3
Probabilities:  [0.67, 0, 0.33, 0]  (67% chance of 'b', 33% chance of 'd')
```

### Visual Example

Let's manually build a tiny bigram table from 3 names:

In [12]:
# Manual bigram example with 3 names
example_names = ['sam', 'max', 'alex']

print("Building bigram counts from:", example_names)
print("\nAdding special tokens:")
for name in example_names:
    print(f"  <S>{name}<E>")

print("\nBigram pairs (current → next):")
for name in example_names:
    # Use list of tokens, not string concatenation
    tokens = ['<S>'] + list(name) + ['<E>']
    print(f"  {name}:", end=" ")
    for i in range(len(tokens) - 1):
        print(f"{tokens[i]}→{tokens[i+1]}", end=", ")
    print()

# Count them
from collections import defaultdict
bigram_counts = defaultdict(lambda: defaultdict(int))

for name in example_names:
    # Use list of tokens, not string concatenation
    tokens = ['<S>'] + list(name) + ['<E>']
    for i in range(len(tokens) - 1):
        curr = tokens[i]
        next_char = tokens[i+1]
        bigram_counts[curr][next_char] += 1

print("\nBigram count table:")
for curr in sorted(bigram_counts.keys()):
    print(f"  After '{curr}': ", end="")
    for next_char, count in sorted(bigram_counts[curr].items()):
        print(f"{next_char}:{count}", end="  ")
    print()

print("\n💡 Key insight: More frequent pairs = higher probability!")

Building bigram counts from: ['sam', 'max', 'alex']

Adding special tokens:
  <S>sam<E>
  <S>max<E>
  <S>alex<E>

Bigram pairs (current → next):
  sam: <→S, S→>, >→s, s→a, a→m, m→<, <→E, E→>, 
  max: <→S, S→>, >→m, m→a, a→x, x→<, <→E, E→>, 
  alex: <→S, S→>, >→a, a→l, l→e, e→x, x→<, <→E, E→>, 

Bigram count table:
  After '<': E:3  S:3  
  After '>': a:1  m:1  s:1  
  After 'E': >:3  
  After 'S': >:3  
  After 'a': l:1  m:1  x:1  
  After 'e': x:1  
  After 'l': e:1  
  After 'm': <:1  a:1  
  After 's': a:1  
  After 'x': <:2  

💡 Key insight: More frequent pairs = higher probability!


## Part 3: Loss Function - The Teacher

### What is Loss?

The **loss function** is like a teacher grading the model:
- **Low loss** = Good prediction! 🎉
- **High loss** = Bad prediction! 😞

### Cross-Entropy Loss for Character Prediction

In our makemore bigram model:
- **Input:** Current character (e.g., 'a')
- **Goal:** Predict next character in name
- **Output:** Probability distribution over all 28 possible characters

**How loss is calculated:**
```
Example: Name is "anna"
Current char: 'a'
True next char: 'n' (character we want to predict)

Model outputs 28 probabilities (one for each character):
  P(<S>) = 0.01
  P(<E>) = 0.03
  P('a') = 0.05
  ...
  P('n') = 0.25  ← This is the correct one!
  P('o') = 0.04
  ...
  (all probabilities sum to 1.0)

Loss = -log(P('n')) = -log(0.25) = 1.39
```

**Why negative log?**
- If model predicts P('n') = 1.0 (100% confident): loss = -log(1.0) = 0 ✅ Perfect!
- If model predicts P('n') = 0.5 (50% confident): loss = -log(0.5) = 0.69 ✓ OK
- If model predicts P('n') = 0.01 (1% confident): loss = -log(0.01) = 4.61 ✗ Bad!

### Training = Minimize Loss Across All Examples

We train on thousands of (current, next) pairs:
- (<S>, 'a'), (<S>, 'e'), (<S>, 'm'), ... (name starts)
- ('a', 'n'), ('a', 'l'), ('a', 'r'), ... (after 'a')
- ('n', 'n'), ('n', 'a'), ('n', 'e'), ... (after 'n')
- etc.

**Training adjusts the bigram table** to make correct predictions more likely!

**Random model:** Loss ≈ 3.3
- Guessing uniformly from 28 characters
- Perplexity ≈ 27

**Trained bigram:** Loss ≈ 2.4
- Learned which characters follow each other
- Perplexity ≈ 11 (narrowed down to ~11 likely choices!)

### Visualizing Loss During Training

As we train, loss decreases:
```
Iteration 0:   Loss = 3.30 (random guessing)
Iteration 10:  Loss = 2.85 (starting to learn)
Iteration 50:  Loss = 2.50 (learned common patterns)
Iteration 100: Loss = 2.40 (converged!)
```

In [13]:
# Manual loss calculation example
import math

print("Example: After 'a', true next char is 'n'\n")

# Good model (confident)
prob_good = 0.60
loss_good = -math.log(prob_good)
print(f"Good model predicts 'n' with probability {prob_good:.2f}")
print(f"  Loss = -log({prob_good:.2f}) = {loss_good:.2f}  ✓ LOW\n")

# Bad model (unconfident)
prob_bad = 0.01
loss_bad = -math.log(prob_bad)
print(f"Bad model predicts 'n' with probability {prob_bad:.2f}")
print(f"  Loss = -log({prob_bad:.2f}) = {loss_bad:.2f}  ✗ HIGH\n")

# Random baseline
prob_random = 1/27  # 27 characters in vocab
loss_random = -math.log(prob_random)
print(f"Random model (uniform distribution over 27 chars)")
print(f"  Loss = -log({prob_random:.4f}) = {loss_random:.2f}  (baseline)\n")

print("💡 Training goal: Make loss go from ~3.3 down to ~2.4!")

Example: After 'a', true next char is 'n'

Good model predicts 'n' with probability 0.60
  Loss = -log(0.60) = 0.51  ✓ LOW

Bad model predicts 'n' with probability 0.01
  Loss = -log(0.01) = 4.61  ✗ HIGH

Random model (uniform distribution over 27 chars)
  Loss = -log(0.0370) = 3.30  (baseline)

💡 Training goal: Make loss go from ~3.3 down to ~2.4!


## Part 4: Building the Bigram Dataset

Before we can train, we need to create (current_char, next_char) pairs from our names.

**Example:**
```
Name: "sam"
With tokens: <S> s a m <E>

Bigram pairs:
  <S> → s
  s → a
  a → m
  m → <E>
```

In [ ]:
# Build bigram dataset
def build_bigram_dataset(names, tokenizer):
    """
    Create bigram training data from names.

    Returns:
        xs: tensor of current character IDs
        ys: tensor of next character IDs
    """
    xs, ys = [], []

    for name in names:
        # Add special tokens
        chars = ['<S>'] + list(name) + ['<E>']

        # Create bigram pairs
        for i in range(len(chars) - 1):
            curr_char = chars[i]
            next_char = chars[i + 1]

            curr_id = tokenizer.char2id[curr_char]
            next_id = tokenizer.char2id[next_char]

            xs.append(curr_id)
            ys.append(next_id)

    return torch.tensor(xs), torch.tensor(ys)

# Create dataset
xs, ys = build_bigram_dataset(names, tokenizer)

print(f"Dataset size: {len(xs):,} bigram pairs")
print(f"\nFirst 10 examples:")
for i in range(10):
    curr_char = tokenizer.id2char[xs[i].item()]
    next_char = tokenizer.id2char[ys[i].item()]
    print(f"  '{curr_char}' → '{next_char}'")

## Part 5: Training a Bigram Model

### Architecture

Our bigram model is VERY simple:
```python
logits_table: (vocab_size, vocab_size) tensor

Forward pass:
  current_char_id = 5  # e.g., 'a'
  logits = logits_table[5]  # Get row 5 → predictions for what comes after 'a'
  probs = softmax(logits)  # Convert to probabilities
```

That's it! Just a lookup table!

### What are logits?

**Logits** = unnormalized scores (before softmax)

Example:
```
Logits:  [2.1, 0.5, -1.2, 3.4, 0.8]
          ↓ softmax (normalizes to sum to 1)
Probs:   [0.18, 0.04, 0.01, 0.67, 0.10]
```

**Why use logits?**
- Easier to optimize (unconstrained)
- More numerically stable

### Training Loop

1. **Forward**: Get logits for current character
2. **Loss**: Compute cross-entropy with true next character
3. **Backward**: Compute gradients
4. **Update**: Adjust logits to reduce loss

## Exercise 3: Implement Bigram Model

### Your Task

Implement a simple bigram model:
1. Create a (vocab_size × vocab_size) parameter tensor (the logits table)
2. Implement forward pass: lookup logits for given character IDs

### Hints
- Use `nn.Parameter()` to make tensor trainable
- Initialize with zeros or small random values
- Forward pass is just indexing: `self.logits[idx]`

In [ ]:
class BigramModel(nn.Module):
    """Simple bigram language model using a lookup table."""

    def __init__(self, vocab_size):
        super().__init__()
        # Create logits table (vocab_size × vocab_size)
        self.logits = nn.Parameter(torch.zeros((vocab_size, vocab_size)))

    def forward(self, idx):
        """
        Get logits for given character IDs.

        Args:
            idx: (N,) tensor of current character IDs
        Returns:
            logits: (N, vocab_size) tensor of scores for next character
        """
        # Look up logits for each character ID
        return self.logits[idx]

# Create model
model = BigramModel(tokenizer.vocab_size)
print(f"Model created!")
print(f"Logits table shape: {model.logits.shape}")
print(f"Number of parameters: {model.logits.numel():,}")

# Test forward pass
test_ids = torch.tensor([0, 1, 2])  # 3 characters
test_logits = model(test_ids)
print(f"\nTest forward pass:")
print(f"  Input shape: {test_ids.shape}")
print(f"  Output shape: {test_logits.shape}")
print(f"✓ Forward pass works!")

## Exercise 2: Calculate Loss on Random Model

### Understanding Loss with Our Bigram Model

Now that we have a model (randomly initialized), let's see what loss looks like BEFORE training.

Our model has a 28×28 logits table, currently all zeros. After softmax, each character has equal probability (1/28 ≈ 0.036).

Let's calculate loss for a few examples from our dataset:

### Your Task

Before we train, let's see what loss looks like with a **random (untrained) model**.

1. Get logits from random model for a few example bigrams
2. Convert logits to probabilities with softmax
3. Calculate loss for each example
4. See average loss ≈ 3.3 (baseline)

This shows us where we START before training!

### Hints
- Use the model you just created
- Get logits: `model(torch.tensor([current_char_id]))`
- Convert to probs: `F.softmax(logits, dim=-1)`
- Loss = -log(prob of correct next character)

In [ ]:
# Calculate loss on random model BEFORE training

print("Testing loss on random (untrained) model:\n")

# Get a few examples from our dataset
num_examples = 10
sample_indices = torch.randint(0, len(xs), (num_examples,))

model.eval()
total_loss = 0

for idx in sample_indices:
    curr_id = xs[idx].item()
    next_id = ys[idx].item()

    curr_char = tokenizer.id2char[curr_id]
    next_char = tokenizer.id2char[next_id]

    # Get model prediction
    with torch.no_grad():
        logits = model(torch.tensor([curr_id]))[0]
        probs = F.softmax(logits, dim=0)
        prob_correct = probs[next_id].item()

    # Calculate loss
    loss = -math.log(prob_correct)
    total_loss += loss

    print(f"'{curr_char}' → '{next_char}': P = {prob_correct:.4f}, Loss = {loss:.2f}")

avg_loss = total_loss / num_examples
print(f"\nAverage loss on random model: {avg_loss:.2f}")
print(f"Expected: ~3.3 (log(28) since 28 characters)")
print(f"\n💡 After training, we'll reduce this to ~2.4!")

## Exercise 4: Implement Training Loop

### Your Task

Train the bigram model:
1. Create model and optimizer
2. Loop for N iterations:
   - Forward pass: get logits
   - Compute loss: cross_entropy(logits, targets)
   - Backward pass: compute gradients
   - Update: optimizer.step()
3. Track and plot loss

### Expected Results
- Initial loss: ~3.3 (random guessing from 27 chars)
- Final loss: ~2.4 (learned bigram patterns!)
- Training takes ~10 seconds

### Hints
- Use `F.cross_entropy(logits, targets)` for loss
- Use `torch.optim.AdamW` optimizer
- Learning rate: 50 works well (yes, 50!)

In [ ]:
# Create model
model = BigramModel(tokenizer.vocab_size)

# Create optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=50)

# Training loop
num_iterations = 100
losses = []

for i in range(num_iterations):
    # Forward pass
    logits = model(xs)  # (N, vocab_size)

    # Compute loss
    loss = F.cross_entropy(logits, ys)
    losses.append(loss.item())

    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print progress
    if i % 10 == 0 or i == num_iterations - 1:
        print(f"Iteration {i:3d}: loss = {loss.item():.4f}")

print(f"\n✓ Training complete!")
print(f"  Initial loss: {losses[0]:.4f}")
print(f"  Final loss:   {losses[-1]:.4f}")
print(f"  Improvement:  {losses[0] - losses[-1]:.4f}")

### Plot Training Loss

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(losses, linewidth=2)
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Bigram Model Training Loss', fontsize=14)
plt.grid(True, alpha=0.3)
plt.axhline(y=losses[0], color='r', linestyle='--', alpha=0.5, label=f'Initial: {losses[0]:.2f}')
plt.axhline(y=losses[-1], color='g', linestyle='--', alpha=0.5, label=f'Final: {losses[-1]:.2f}')
plt.legend()
plt.show()

print("💡 Loss decreased! Model learned bigram patterns!")

## Part 6: Visualizing the Bigram Table

Now let's see what the model learned! We'll visualize the bigram probability table as a heatmap.

**What to look for:**
- Bright spots = high probability (common bigrams)
- Dark spots = low probability (rare bigrams)
- Patterns in rows show what typically follows each character

## Exercise 5: Visualize Bigram Table

### Your Task

Create a heatmap of the learned bigram probabilities:
1. Convert logits to probabilities using softmax
2. Create heatmap with matplotlib or seaborn
3. Label axes with characters

### Hints
- Use `F.softmax(model.logits, dim=1)` to get probabilities
- Use `plt.imshow()` or `sns.heatmap()`
- Set character labels for x and y axes

In [ ]:
# Get probability table
probs = F.softmax(model.logits, dim=1).detach().numpy()

# Create character labels
chars = [tokenizer.id2char[i] for i in range(tokenizer.vocab_size)]

# Create heatmap
plt.figure(figsize=(14, 12))
sns.heatmap(probs, cmap='Blues', xticklabels=chars, yticklabels=chars,
            square=True, cbar_kws={'label': 'Probability'})
plt.xlabel('Next Character', fontsize=12)
plt.ylabel('Current Character', fontsize=12)
plt.title('Bigram Probability Table (Trained Model)', fontsize=14)
plt.tight_layout()
plt.show()

print("💡 Bright spots = common bigrams!")
print("   Dark spots = rare bigrams!")

### Text-Based Grid Visualization

Let's also view the bigram table as a text grid for easier inspection of specific values:

In [ ]:
def print_bigram_grid(model, tokenizer, top_n=10):
    """
    Print bigram probability table as text grid.
    
    Args:
        model: trained bigram model
        tokenizer: character tokenizer
        top_n: show top N most common starting characters
    """
    # Get probability table
    probs = F.softmax(model.logits, dim=1).detach()
    
    # Get characters
    chars = [tokenizer.id2char[i] for i in range(tokenizer.vocab_size)]
    
    # Find most common starting characters (excluding special tokens)
    start_probs = probs[tokenizer.start_id]
    _, top_indices = torch.topk(start_probs, top_n + 2)  # +2 to skip special tokens
    
    # Filter out special tokens and get top_n
    display_chars = []
    for idx in top_indices:
        if idx.item() > 1:  # Skip <S> and <E>
            display_chars.append(idx.item())
            if len(display_chars) >= top_n:
                break
    
    # Also include special tokens
    row_chars = [0, 1] + display_chars  # <S>, <E>, then top chars
    col_chars = [0, 1] + display_chars
    
    # Print header
    print("\n" + "="*70)
    print("BIGRAM PROBABILITY TABLE (showing most common characters)")
    print("="*70)
    print("Format: Each row shows P(next_char | current_char)")
    print("        Probabilities shown as percentages")
    print("="*70 + "\n")
    
    # Print column headers
    print("        Next char")
    print("Curr   ", end="")
    for col_idx in col_chars:
        col_char = chars[col_idx]
        print(f"{col_char:>5}", end="")
    print()
    
    # Print separator
    print("     " + "-" * (5 * len(col_chars) + 5))
    
    # Print rows
    for row_idx in row_chars:
        row_char = chars[row_idx]
        print(f"  {row_char:>2}  |", end="")
        
        for col_idx in col_chars:
            prob = probs[row_idx, col_idx].item()
            # Format as percentage
            if prob >= 0.10:
                print(f"{prob*100:5.1f}", end="")  # Show 1 decimal for high probs
            elif prob >= 0.01:
                print(f"{prob*100:5.2f}", end="")  # Show 2 decimals for medium probs
            else:
                print(f"  .  ", end="")  # Show . for very small probs
        print()
    
    print("\n" + "="*70)
    print("Legend: Numbers are percentages (%), '.' means <1%")
    print("        Row = current char, Column = next char")
    print("="*70 + "\n")

# Show the grid
print_bigram_grid(model, tokenizer, top_n=8)

print("\n💡 How to read this table:")
print("   - Row 'a', Column 'n': P(next='n' | curr='a')")
print("   - High values = common bigrams")
print("   - '.' = rare bigrams (probability < 1%)")


### Analyze Specific Patterns

Let's look at specific characters and see what the model learned:

In [ ]:
def show_top_next_chars(model, tokenizer, char, top_k=5):
    """Show top-k most likely next characters for given character."""
    char_id = tokenizer.char2id[char]
    logits = model.logits[char_id]
    probs = F.softmax(logits, dim=0)

    # Get top-k
    top_probs, top_indices = torch.topk(probs, top_k)

    print(f"After '{char}', most likely next characters:")
    for i, (prob, idx) in enumerate(zip(top_probs, top_indices)):
        next_char = tokenizer.id2char[idx.item()]
        print(f"  {i+1}. '{next_char}': {prob.item():.2%}")
    print()

# Test with interesting characters
show_top_next_chars(model, tokenizer, 'q')  # Should show 'u'!
show_top_next_chars(model, tokenizer, 'a')
show_top_next_chars(model, tokenizer, 'x')
show_top_next_chars(model, tokenizer, '<S>')  # What starts names?

### Find Most Common Bigrams

In [ ]:
# Get all bigram probabilities
probs = F.softmax(model.logits, dim=1).detach()

# Find top-20 most likely bigrams
flat_probs = probs.flatten()
top_20_values, top_20_indices = torch.topk(flat_probs, 20)

print("Top 20 most common bigrams:")
for i, (prob, idx) in enumerate(zip(top_20_values, top_20_indices)):
    row = idx.item() // tokenizer.vocab_size
    col = idx.item() % tokenizer.vocab_size
    curr_char = tokenizer.id2char[row]
    next_char = tokenizer.id2char[col]
    print(f"{i+1:2d}. '{curr_char}' → '{next_char}': {prob.item():.2%}")

## Part 7: Generating Names

Now the fun part - let's use our trained model to generate new names!

### Algorithm

1. Start with `<S>` token
2. Get probabilities for next character
3. Sample from probability distribution
4. Repeat until we sample `<E>` token

### Sampling Strategies

**Greedy** (always pick highest probability):
- Deterministic
- Always generates same output
- Can be repetitive

**Random sampling** (sample from distribution):
- Stochastic (different each time)
- More diverse outputs
- Can generate rare but valid names

In [ ]:
def generate_name(model, tokenizer, max_length=20, sample=True, seed=None):
    """
    Generate a name from the bigram model.

    Args:
        model: trained bigram model
        tokenizer: character tokenizer
        max_length: maximum name length
        sample: if True, sample from distribution; if False, use greedy
        seed: random seed for reproducibility
    Returns:
        generated name (string)
    """
    if seed is not None:
        torch.manual_seed(seed)

    model.eval()

    # Start with <S> token
    curr_id = tokenizer.start_id
    name_ids = []

    for _ in range(max_length):
        # Get logits for current character
        with torch.no_grad():
            logits = model(torch.tensor([curr_id]))[0]  # (vocab_size,)
            probs = F.softmax(logits, dim=0)

        # Sample next character
        if sample:
            next_id = torch.multinomial(probs, num_samples=1).item()
        else:
            next_id = torch.argmax(probs).item()

        # Stop if we hit <E> token
        if next_id == tokenizer.end_id:
            break

        name_ids.append(next_id)
        curr_id = next_id

    # Decode
    name = tokenizer.decode(name_ids)
    return name

# Generate 20 random names
print("Generated names (random sampling):\n")
for i in range(20):
    name = generate_name(model, tokenizer, sample=True)
    print(f"{i+1:2d}. {name}")

# Try greedy sampling
print("\nGreedy sampling (always picks most likely):")
greedy_name = generate_name(model, tokenizer, sample=False)
print(f"  {greedy_name}")

### Evaluate Generated Names

In [ ]:
# Generate 100 names and analyze
generated = [generate_name(model, tokenizer) for _ in range(100)]

# Check how many are in training set (overfitting?)
in_train = sum(1 for name in generated if name in names)
print(f"Generated 100 names:")
print(f"  Average length: {sum(len(n) for n in generated) / len(generated):.1f}")
print(f"  Exact matches with training data: {in_train}% (lower is better!)")
print(f"  Unique names generated: {len(set(generated))}")

# Show some interesting ones
print(f"\nSome interesting generated names:")
unique_names = list(set(generated) - set(names))[:10]
for name in unique_names:
    print(f"  - {name}")

## Part 8: Model Evaluation

Let's measure how well our model performs.

### Metrics

**Loss**: Average negative log-likelihood
- Lower is better
- Our model: ~2.4
- Random baseline: ~3.3

**Perplexity**: exp(loss)
- Interpretable: "On average, model is as confused as picking uniformly from N options"
- Our model: exp(2.4) ≈ 11 (effectively choosing from 11 likely chars)
- Random baseline: exp(3.3) ≈ 27 (choosing uniformly from all 27 chars)

In [ ]:
# Evaluate on full dataset
model.eval()
with torch.no_grad():
    logits = model(xs)
    loss = F.cross_entropy(logits, ys)
    perplexity = torch.exp(loss)

print(f"Final Model Performance:")
print(f"  Loss: {loss.item():.4f}")
print(f"  Perplexity: {perplexity.item():.2f}")
print(f"\nInterpretation:")
print(f"  Model is as confused as picking uniformly from ~{perplexity.item():.0f} characters")
print(f"  (vs. 27 for random guessing)")
print(f"\n✓ Model learned bigram patterns!")

## Summary

### What You've Learned

✅ **Tokenization**: Convert text ↔ numbers for neural networks
- Built character-level tokenizer from scratch
- Added special tokens `<S>` and `<E>`

✅ **Loss as Teacher**: Cross-entropy loss guides learning
- Low loss = good predictions
- High loss = bad predictions
- Training = minimize loss

✅ **Bigram Model**: Simplest language model
- Predicts next character from current character
- Just a lookup table (vocab_size × vocab_size)
- Learns which character pairs are common

✅ **Visualization**: Interpreted learned patterns
- Heatmap of bigram probabilities
- Identified common patterns (e.g., 'q' → 'u')

✅ **Generation**: Sampled from learned distribution
- Generated plausible new names
- Used random sampling for diversity

### Limitations of Bigram Model

**Only 1 character of context**
- Can't learn patterns like "consonant after vowel"
- Can't remember start of name

**Example:**
```
Name so far: "Elizab"
Next char: ?

Bigram model only sees: "b" → predicts next
  (Doesn't know we're in a long name, doesn't know vowel pattern)

Better model would see: "Elizab" → predicts next
  (Can use more context for better predictions)
```

**Performance:**
- Loss: ~2.4
- Perplexity: ~11

### Next Steps

**Lab 6 Part 2: NanoGPT** (coming next)
- Use more context (8-64 characters)
- Attention mechanism to focus on relevant parts
- Better performance (perplexity ~5)
- This is how GPT works!

**Key insight:** Bigrams are foundation for understanding all language models. You now understand the core concepts:
1. Convert text to tokens
2. Learn probability distributions
3. Minimize loss (maximize likelihood)
4. Sample to generate text

Everything else (MLP, RNN, Transformer) is just about **using more context better**!

---

**🎉 Congratulations! You've built and trained your first language model!**